In [1]:
import json

old_file = "./../msvamp_multicolumn_multilang.jsonl"
new_file = "./msvamp_multicolumn_multilang.jsonl"
merged_file = "msvamp_multicolumn_multilang_interim.jsonl"

with open(old_file, "r", encoding="utf-8") as f1, open(new_file, "r", encoding="utf-8") as f2, open(merged_file, "w", encoding="utf-8") as fout:
    for line1, line2 in zip(f1, f2):
        d1 = json.loads(line1)
        d2 = json.loads(line2)

        # Drop 'question' from the second file to avoid duplication
        d2.pop("m_query", None)

        # Merge dictionaries
        merged = {**d1, **d2}

        fout.write(json.dumps(merged, ensure_ascii=False) + "\n")

print("✅ Merged successfully into", merged_file)


✅ Merged successfully into msvamp_multicolumn_multilang_interim.jsonl


In [2]:
import json
from datasets import load_dataset
merged_file = "msvamp_multicolumn_multilang_interim.jsonl"
merged_data = []

with open(merged_file, "r", encoding="utf-8") as f:
    for line in f:
        merged_data.append(json.loads(line))

print(f"Loaded {len(merged_data)} lines from merged.jsonl")

# Step 2: Load the three MGSM datasets
langs = ["fr", "es", "zh"]
datasets = {lang: load_dataset("Mathoctopus/MSVAMP", lang, split="test") for lang in langs}

# Step 3: Verify all have the same length
n = len(merged_data)
assert all(len(ds) == n for ds in datasets.values()), "Mismatch in dataset lengths!"

# Step 4: Merge side-by-side
for i in range(n):
    for lang in langs:
        merged_data[i][f"m_query_{lang}"] = datasets[lang][i]["m_query"]

# Step 5: Write out to new JSONL
with open("msvamp_multicolumn_multilang_final.jsonl", "w", encoding="utf-8") as fout:
    for row in merged_data:
        fout.write(json.dumps(row, ensure_ascii=False) + "\n")

print("✅ Created merged_final.jsonl successfully")


Loaded 1000 lines from merged.jsonl
✅ Created merged_final.jsonl successfully


In [6]:
import json

# Load your JSON file
with open("translations_prompt_msvamp.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Add a new entry for Hindi
data["en"] = {
    "instruction": "You are a math assistant. Give the solution to the problem in steps. Give final numerical answer."
}
data["fr"] = {
    "instruction": "Vous êtes un assistant de mathématiques. Donnez la solution du problème en étapes. Donnez la réponse numérique finale."
}
data["es"] = {
    "instruction": "Eres un asistente de matemáticas. Da la solución al problema en pasos. Da la respuesta numérica final."
}
data["zh"] = {
    "instruction": "你是一名数学助手。请分步骤给出题目的解答。给出最终的数值答案。"
}
data["ko"] = {
    "instruction": "당신은 수학 보조자입니다. 문제의 풀이를 단계별로 제시하세요. 최종 수치 답을 제시하세요."
}
data["hi"] = {
    "instruction": "आप एक गणित सहायक हैं। समस्या का समाधान चरणों में दें। अंतिम संख्यात्मक उत्तर दें।"
}

# Save it back
with open("translations_prompt_msvamp.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)


In [7]:
# TRANSLATION READY/ PROMPTS READY

In [1]:
import pandas as pd
mapping = pd.read_csv('fasttext_nllb_mapping_98.csv')
mapping = mapping[(mapping.Duplicate_Flag == False) & (mapping.Missing_Flag == False)][['fasttext_code','NLLB_Code']]
mapping_dict = dict(zip(mapping['fasttext_code'], mapping['NLLB_Code']))
mapping_dict['zh'] = 'zh_Hans'
print(len(mapping_dict))

101


In [4]:
num_samples = 20

from vllm import LLM, SamplingParams
model_name = "allenai/OLMo-2-1124-7B-Instruct"
llm = LLM(
    model=model_name,
    dtype="float16",
    gpu_memory_utilization=0.85  # safe for single GPU
)
sampling_params = SamplingParams(
    temperature=0.7,
    top_k = 40,
    max_tokens=1024,
    n = num_samples
)




# SELF CONSISTENCY


import pandas as pd
mapping = pd.read_csv('fasttext_nllb_mapping_98.csv')
mapping = mapping[(mapping.Duplicate_Flag == False) & (mapping.Missing_Flag == False)][['fasttext_code','NLLB_Code']]
mapping_dict = dict(zip(mapping['fasttext_code'], mapping['NLLB_Code']))
mapping_dict['zh'] = 'zh_Hans'
print(len(mapping_dict))
from datasets import *
ds = load_dataset("json", data_files="msvamp_multicolumn_multilang_final.jsonl", split="train")

ref = load_dataset("Mathoctopus/MSVAMP", "en", split="test")
references = [ex["response"] for ex in ref]

import re
import numpy
def extract_final_answer(text: str):
    # Regex matches numbers with optional commas, decimals, and negatives
    numbers = re.findall(r'-?\d[\d,]*\.?\d*', text)
    if not numbers:
        return None
    return str((numbers[-1]))  # return the last number

def compute_accuracy(predictions, gold_answers):
    correct = 0
    total = len(predictions)
    for pred_text, gold in zip(predictions, gold_answers):
        pred_num = extract_final_answer(pred_text)
        if pred_num is not None:
            if pred_num.endswith('.'):
                pred_num = pred_num[:-1]
            pred_num_norm = pred_num.replace(",", "")
        else:
            pred_num_norm = None
        gold_norm = str(gold)
        if pred_num_norm is not None:
            if float(pred_num_norm) == float(gold_norm):
                correct += 1
    accuracy = correct / total
    return accuracy

import json
with open("translations_prompt_msvamp.json", "r", encoding="utf-8") as f:
    translations = json.load(f)
multi_lang_prompt = {}

for lang_code, parts in translations.items():
    # Compose the Step-by-Step example by joining steps + answer (mirrors the 'en' example)
    example_steps = (parts.get("steps", "").strip() + " " + parts.get("answer", "").strip()).strip()
    # Build the prompt template with a {} placeholder for the new question
    prompt = f"""{parts.get("instruction", "").strip()}
    Question: {parts.get("question", "").strip()}
    Reason(steps):  
    Final Answer:"""
    multi_lang_prompt[lang_code] = prompt


from collections import Counter

def majority_vote(output_texts):
    # For each prompt, outputs are repeated num_samples times in order
    results = []
    for i in range(0, len(output_texts), num_samples):
        group = output_texts[i:i+num_samples]
        # Extract the final numeric answer
        answers = [extract_final_answer(text) for text in group if extract_final_answer(text) is not None]
        if answers:
            most_common = Counter(answers).most_common(1)[0][0]
        else:
            most_common = None
        results.append(most_common)
    return results


results = {}
for lang in list(mapping_dict.keys()):
    try:
        print(lang)
        if lang in ['bn','de','en','es','fr','ja','ru','sw','th','zh']:
            temp = load_dataset("Mathoctopus/MSVAMP", lang, split="test")
            prompts = [multi_lang_prompt[lang].format(ex["m_query"]) for ex in temp]
        else:
            prompts = [multi_lang_prompt[lang].format(ex[f"m_query_{lang}"]) for ex in ds]
        batch_size = 32
        all_outputs = []
    
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i:i + batch_size]
            outputs = llm.generate(batch_prompts, sampling_params)
            for out in outputs:
                seq_texts = [res.text.strip() for res in out.outputs]
                all_outputs.extend(seq_texts)
        # Apply majority voting for self-consistency
        final_preds = majority_vote(all_outputs)
        final_preds = [str(i) for i in final_preds]
        results[lang] = compute_accuracy(final_preds, references)
        print(results)
    except Exception as e:
        print(e)
    
print("="*60)
print(results)


101


In [5]:
for lang in list(mapping_dict.keys()):
    print(lang,end =' ')

als arz ast azb ceb ckb ilo lmo mai min scn vec war yue af am as ba be bn bo bs bg ca cs cy da de el en eo et eu fi fr gd ga gl gn gu ht he hi hr hu hy id is it jv ja kn ka kk km ky ko lo li lt lb ml mr mk mt my nl nn oc pa pl pt ro ru sa si sk sl sd so es sc sr su sv ta tt te tg tl th tk tr ug uk ur vi yo sw yi zh 